# R² levers

Each arm changes one thing on the older stock model: outlier drop, mutual-information median, 100 trees. Numbers come from `runs/r2_levers.jsonl`. This notebook does not refit.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from experiments.blocks import report

raw = [json.loads(line) for line in (ROOT / "runs/r2_levers.jsonl").read_text().splitlines() if line.strip()]
order = [
    "baseline_mi_stock",
    "A_all_post_cull",
    "C_no_row_drop",
    "D_seed_mean",
    "D_xgb_ridge",
    "B_search_prior",
]
by_arm = {r["arm"]: r for r in raw if r["arm"] in order}
base = by_arm["baseline_mi_stock"]["cv_mean"]
arms = pd.DataFrame([
    {
        "arm": name,
        "n": by_arm[name]["n_train"],
        "p": by_arm[name]["n_features"],
        "r2": by_arm[name]["cv_mean"],
        "std": by_arm[name]["cv_std"],
        "delta": by_arm[name]["cv_mean"] - base,
        "note": by_arm[name].get("note") or "",
    }
    for name in order
])
report(arms, "One change versus the stock baseline", bar="r2")


arm,n,p,r2,std,delta,note,
baseline_mi_stock,1199,413,0.480,0.028,0.000,,
A_all_post_cull,1199,825,0.472,0.037,-0.008,,
C_no_row_drop,1212,413,0.474,0.036,-0.006,,
D_seed_mean,1199,413,0.480,0.028,-0.000,"seeds=(42, 43, 44)",
D_xgb_ridge,1199,413,0.457,0.069,-0.023,"mean(stock XGB, Ridge top-50 |corr|)",
B_search_prior,1199,413,0.521,0.049,0.041,"{'max_depth': 4, 'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_weight': 7}",


## Tree-size grid

24 settings on the same mutual-information columns. The prior sits in the block above.


In [2]:
import ast
grid_rows = []
for r in raw:
    if r["arm"] != "B_search_grid":
        continue
    params = ast.literal_eval(r["note"])
    grid_rows.append({
        "depth": params["max_depth"],
        "subsample": params["subsample"],
        "colsample": params["colsample_bytree"],
        "min_child_weight": params["min_child_weight"],
        "r2": r["cv_mean"],
        "delta": r["cv_mean"] - base,
    })
grid = pd.DataFrame(grid_rows).sort_values("r2", ascending=False)
report(grid, "Grid, best first", bar="r2")


depth,subsample,colsample,min_child_weight,r2,delta,
5,0.600,0.800,5,0.523,0.043,
5,0.600,0.600,5,0.522,0.042,
4,0.600,0.800,5,0.520,0.040,
5,0.600,0.600,10,0.520,0.040,
4,0.600,0.600,5,0.520,0.040,
4,0.600,0.600,10,0.518,0.038,
3,0.600,0.600,5,0.517,0.037,
4,0.800,0.800,10,0.516,0.036,
4,0.600,0.800,10,0.514,0.034,
5,0.600,0.800,10,0.513,0.034,
